# 4 Train multilingual mBERT emotion classifier

This notebook builds the emotion classifier/evaluator used later to evalute whether Qwen's English translations preserve the original SMP emotion.

Qwen outputs are intentionally not used to train or fine-tune this model to avoid the dependency on the translation.

### Goal
Fine-tune `google-bert/bert-base-multilingual-cased` to predict:

- `joy`
- `anger`
- `sadness`
- `fear`
- `surprise`
- `neutral`

Training uses native text from both languages:

- **Chinese:** SMP2020-EWECT `usual` posts from `usual_splits.csv`
- **English:** GoEmotions Reddit comments


### Steps
1. Load `usual_splits.csv`
2. Create the six class labels `joy`, `anger`, `sadness`, `fear`, `surprise`, `neutral`
3. Load/filter GoEmotions to the same six class labels
4. Build Chinese + English train/validation/test sets
5. Create majority-class baseline model
6. Create a fine-tuned model with six-class head
7. Select the best checkpoint using validation macro-F1
8. Validate held-out Chinese and English performance
9. Freeze final calssifier

#### Outputs

- `models/mbert_emotion_6class/`
- `data/results/mbert_baseline_metrics.csv`
- `data/results/mbert_test_metrics.csv`
- `data/results/mbert_per_class_metrics.csv`
- `data/results/mbert_training_summary.json`

## 0. Install and import dependencies

In [1]:
%pip install -q -U "transformers>=4.45,<6" datasets accelerate scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [30]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

RANDOM_SEED = 266
set_seed(RANDOM_SEED)

#https://huggingface.co/google-bert/bert-base-multilingual-cased
MODEL_NAME = "google-bert/bert-base-multilingual-cased"
MAX_LENGTH = 128

current_folder = Path.cwd()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder

processed_folder = project_root / "data" / "processed"
results_folder = project_root / "data" / "results"
model_folder = project_root / "models" / "mbert_emotion_6class"


results_folder.mkdir(parents=True, exist_ok=True)
model_folder.mkdir(parents=True, exist_ok=True)

usual_splits_path = processed_folder / "usual_splits.csv"

print("Project root:", project_root)
print("Input:", usual_splits_path)
print("Model output:", model_folder)
print("Results output:", results_folder)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Project root: /content
Input: /content/data/processed/usual_splits.csv
Model output: /content/models/mbert_emotion_6class
Results output: /content/data/results
CUDA available: True
GPU: Tesla T4


## 1. Load SMP usual data

Note: SMP rows overlapping HADQAET must not be used to fine-tune mBERT because it will later act as human-annotated validation resource so we need to filter `split == "train" AND in_hadqaet == False`



In [4]:
print("Path:", usual_splits_path)
print("Exists:", usual_splits_path.exists())
print("Size:", usual_splits_path.stat().st_size, "bytes")

with open(usual_splits_path, "rb") as f:
    first_bytes = f.read(100)

print("First 100 raw bytes:")
print(first_bytes)

Path: /content/data/processed/usual_splits.csv
Exists: True
Size: 6811139 bytes
First 100 raw bytes:
b'\xef\xbb\xbfpost_id,split,text,label,in_hadqaet,mbert_train_eligible,has_slang_simple,slang_terms,slang_defin'


In [5]:
smp = pd.read_csv(usual_splits_path, encoding="utf-8-sig")

required_columns = {"post_id", "split", "text", "label", "in_hadqaet"}
missing = required_columns - set(smp.columns)

if missing:
  raise ValueError(f"Missing columns: {missing}")

print("Rows:", len(smp))
print("\nSplit counts:")
print(smp["split"].value_counts())
print("\nHADQAET-overlapping rows:", int(smp["in_hadqaet"].astype(bool).sum()))
print("mBERT-train-eligible rows:", int(smp["mbert_train_eligible"].astype(bool).sum()))

Rows: 34766

Split counts:
split
train    27864
eval      3457
test      3445
Name: count, dtype: int64

HADQAET-overlapping rows: 5977
mBERT-train-eligible rows: 21887


## 2. Create six label classes

GoEmotions dataset has 28 lables (27 emotions + neutral). We only need change 3 SMP labels (happy -> joy, angery -> anger, sad -> sadness).

| Shared label | SMP label | GoEmotions label |
|---|---|---|
| joy | happy | joy |
| anger | angry | anger |
| sadness | sad | sadness |
| fear | fear | fear |
| surprise | surprise | surprise |
| neutral | neutral | neutral |

In [6]:
LABELS = ["joy", "anger", "sadness", "fear", "surprise", "neutral"]

label_to_ids = {label: idx for idx, label in enumerate(LABELS)}
ids_to_label = {idx: label for label, idx in label_to_ids.items()}

# Map smp labels to shared labels
SMP_TO_SHARED = {
    "happy": "joy",
    "angry": "anger",
    "sad": "sadness",
    "fear": "fear",
    "surprise": "surprise",
    "neutral": "neutral"
}

smp["shared_label"] = smp["label"].map(SMP_TO_SHARED)

# Validate no unmapped
unmapped_smp = smp.loc[smp["shared_label"].isna(), "label"].unique().tolist()
if unmapped_smp:
    raise ValueError(f"Unmapped SMP labels found: {unmapped_smp}")

print("Shared labels:", LABELS)
print("Label ids:", label_to_ids)


Shared labels: ['joy', 'anger', 'sadness', 'fear', 'surprise', 'neutral']
Label ids: {'joy': 0, 'anger': 1, 'sadness': 2, 'fear': 3, 'surprise': 4, 'neutral': 5}


## 3. Prepare native Chinese SMPP train/validation/test sets
- Deduplicate exact text within each split
- Train only on `mbert_train_eligible == True`
- Use SMP `eval` as Chinese validation
- Keep SMP `test` held out for final classifier validation

**Note:** Notebook 2 already keeps duplicate text from crossing split boundaries

In [7]:
smp["normalized_text"] = smp["text"].astype(str).str.strip()

def deduplicate_text(df):
    return (
        df
        .drop_duplicates(subset=["normalized_text"], keep="first")
        .copy()
        .reset_index(drop=True)
    )

# Only take mBERT train eligible from the train split
train_raw = deduplicate_text(
    smp[smp["mbert_train_eligible"].astype(bool)]
)
val_raw = deduplicate_text(
    smp[smp["split"] == "eval"]
)
test_raw = deduplicate_text(
    smp[smp["split"] == "test"]
)

assert not train_raw["in_hadqaet"].astype(bool).any()

train_texts = set(train_raw["normalized_text"])
val_texts = set(val_raw["normalized_text"])
test_texts = set(test_raw["normalized_text"])

assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

print("Chinese train:", len(train_raw))
print("Chinese validation:", len(val_raw))
print("Chinese test:", len(test_raw))
print("\nChinese train label distribution:")
print(train_raw["shared_label"].value_counts())

Chinese train: 21375
Chinese validation: 3360
Chinese test: 3360

Chinese train label distribution:
shared_label
neutral     5702
anger       5604
joy         3997
sadness     3720
surprise    1491
fear         861
Name: count, dtype: int64


## 4. Load English GoEmotions data

From Gimini overview "The GoEmotions - Kaggle dataset features 58,000 curated Reddit comments labeled with 28 boolean emotion categories (27 fine-grained emotions plus Neutral). It is a multi-label dataset where a single comment can be tagged with one or multiple labels."

https://huggingface.co/datasets/google-research-datasets/go_emotions

In [8]:
go = load_dataset(
    "google-research-datasets/go_emotions",
    "simplified",
)

go_label_names = go["train"].features["labels"].feature.names

print("GoEmotions label count:", len(go_label_names))
print(go_label_names)

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

GoEmotions label count: 28
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [9]:
def filter_goemotions(hf_split):
    rows = []
    for example in hf_split:
        label_names = [
            go_label_names[label_id]
            for label_id in example["labels"]
        ]

        if len(label_names) == 1 and label_names[0] in LABELS:
            rows.append({
                "text": example["text"],
                "shared_label": label_names[0],
            })

    return pd.DataFrame(rows)

en_train_raw = filter_goemotions(go["train"])
en_val_raw = filter_goemotions(go["validation"])
en_test_raw = filter_goemotions(go["test"])


print("English train:", len(en_train_raw))
print("English validation:", len(en_val_raw))
print("English test:", len(en_test_raw))
print("\nEnglish train label distribution:")
print(en_train_raw["shared_label"].value_counts())

English train: 16668
English validation: 2044
English test: 2084

English train label distribution:
shared_label
neutral     12823
anger        1025
joy           853
sadness       817
surprise      720
fear          430
Name: count, dtype: int64


## 5. Build multilingual train/validation/test sets

The mBERT training set combines native Chinese SMP posts and native English GoEmotions Reddit comments.

The validation set also combines both languages so checkpoint selection can reflect the multilingual classification task.

The test performance is reported separately for Chinese and English.

In [10]:
def to_classifier_frame(df, language, source):
  output_df = df[["text", "shared_label"]].copy()
  output_df["labels"] = output_df["shared_label"].map(label_to_ids)
  output_df["language"] = language
  output_df["source"] = source

  if output_df["labels"].isna().any():
        raise ValueError(f"Unmapped labels found in {source}")

  return output_df[["text", "shared_label", "labels", "language", "source"]]

In [11]:
# Get df from all Chinese and English sets
cn_train_df = to_classifier_frame(train_raw, "Chinese", "SMP")
cn_val_df = to_classifier_frame(val_raw, "Chinese", "SMP")
cn_test_df = to_classifier_frame(test_raw, "Chinese", "SMP")

en_train_df = to_classifier_frame(en_train_raw, "English", "GoEmotions")
en_val_df = to_classifier_frame(en_val_raw, "English", "GoEmotions")
en_test_df = to_classifier_frame(en_test_raw, "English", "GoEmotions")

# Combine Chinese and English train/validation/test sets,
# For each, shuffle the rows but set random_state to make sure it's reproducible
train_df = pd.concat([cn_train_df, en_train_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

val_df = pd.concat([cn_val_df, en_val_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

test_df = pd.concat([cn_test_df, en_test_df], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Total train:", len(train_df))
print("Total validation:", len(val_df))
print("Total test:", len(test_df))

print("\nTraining counts by language and emotion:")
display(pd.crosstab(train_df["shared_label"], train_df["language"]))

print("\nValidation counts by language and emotion:")
display(pd.crosstab(val_df["shared_label"], val_df["language"]))

Total train: 38043
Total validation: 5404
Total test: 5444

Training counts by language and emotion:


language,Chinese,English
shared_label,,
anger,5604,1025
fear,861,430
joy,3997,853
neutral,5702,12823
sadness,3720,817
surprise,1491,720



Validation counts by language and emotion:


language,Chinese,English
shared_label,,
anger,956,109
fear,145,58
joy,674,106
neutral,712,1592
sadness,619,84
surprise,254,95


## 6. Create majority-class classifier baseline
A simple baseline that predicts the most common training emotion

In [12]:
def majority_baseline(train_df, test_df, dataset_name):
  majority_id = int(train_df["labels"].value_counts().idxmax())

  y_true = test_df["labels"].to_numpy()
  y_pred = np.full(len(test_df), majority_id)

  return {
      "model": "majority_class",
      "dataset": dataset_name,
      "majority_label": ids_to_label[majority_id],
      "accuracy": accuracy_score(y_true, y_pred),
      "macro_f1": f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro"),
  }

In [13]:
baseline_metrics = pd.DataFrame([majority_baseline(cn_train_df, cn_test_df, "SMP Chinese Test"),
                                 majority_baseline(en_train_df, en_test_df, "GoEmotions English Test"),
                                 majority_baseline(train_df, test_df, "Combined Multilingual Test")])
baseline_path = results_folder / "mbert_baseline_metrics.csv"
baseline_metrics.to_csv(baseline_path, index=False)

display(baseline_metrics)
print("Saved:", baseline_path)

,model,dataset,majority_label,accuracy,macro_f1
0,majority_class,SMP Chinese Test,neutral,0.211905,0.058284
1,majority_class,GoEmotions English Test,neutral,0.770633,0.145077
2,majority_class,Combined Multilingual Test,neutral,0.425790,0.099545


Saved: /content/data/results/mbert_baseline_metrics.csv


## 7. Define and fine-tune mBERT


### Tokenize data and add a six-class classification head

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

# Keep only text and labels in the following datasets
# where tex is the input text and labels is the numeric target class used to calculate loss
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]])
val_dataset = Dataset.from_pandas(val_df[["text", "labels"]])

# tokenize train and val datasets
tokenized_train_ds = train_dataset.map(tokenize_function, batched=True)
tokenized_val_ds = val_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# AutoModelForSequenceClassification adds the six-class head that we fine-tune jointly with the encoder
# https://huggingface.co/docs/transformers/v5.23.0/en/model_doc/auto#transformers.AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ids_to_label,
    label2id=label_to_ids,
)


print("Base model:", MODEL_NAME)
print("Number of output classes:", model.config.num_labels)


Map:   0%|          | 0/38043 [00:00<?, ? examples/s]

Map:   0%|          | 0/5404 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Base model: google-bert/bert-base-multilingual-cased
Number of output classes: 6


### Define validation metrics (accuracy and macro F1)
Use macro-F1 because it gives each emotion class equal weight and is therefore more informative than accuracy when the class distribution is imbalanced (which is true in this case)

F1 = 2((precision x recall)/(precision + recall))

In this case macro-F1 = (F1[joy] + F1[anger] + F1[saddness] + F1[fear] + F1[surprise] + F1[neutral])/2

In [17]:
def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    predicted_labels = np.argmax(logits, axis=-1)

    return {"accuracy": accuracy_score(true_labels, predicted_labels),
            "macro_f1": f1_score(true_labels, predicted_labels,
                                 labels=list(range(len(LABELS))),
                                 average="macro", zero_division=0)
            }

### Fine-tune mBERT

**Best configuration:**
- learning rate: 2e-5
- epochs: 3
- weight decay: 0.01
- per device train batch size=16,
- per device eval batch size=32,
- dynamic padding
- best checkpoint selected using validation macro-F1 defined above

Set random seed so it's reproducible

In [20]:
use_bf16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

training_args = TrainingArguments(
    output_dir=str(model_folder / "checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay= 0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=100,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    report_to="none",
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.645244,0.642694,0.777202,0.707455
2,0.507991,0.611190,0.793116,0.730400
3,0.361595,0.664983,0.794782,0.736903


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Best checkpoint: /content/models/mbert_emotion_6class/checkpoints/checkpoint-7134
Best validation macro-F1: 0.7369032923740276


### Validate mBERT
Before deciding ona frozen classifier we report held-out performance separately for
- Chinese SMP test data
- English GoEmotions test data
- Combined multilingual test data

Note that in this part we're still NOT using the Qwen's English translations. That's for the classificaiton in the next part


In [23]:
def predict_output_df(df):
  dataset = Dataset.from_pandas(df[["text", "labels"]], preserve_index=True)
  tokenized_ds = dataset.map(tokenize_function, batched=True)
  output = trainer.predict(tokenized_ds)
  y_true = df["labels"].to_numpy()

  # the shape is [n, 6], argmax(axis=1) to get the column with the highest prabability for each sample
  y_pred = output.predictions.argmax(axis=1)

  return y_true, y_pred

In [31]:
def summarize_test_set(df, dataset_name):
  y_true, y_pred = predict_output_df(df)

  summary = {
      # should be my pride and joy fine tuned mBERT
      "model": "Finetuned-mBERT",
      "dataset": dataset_name,
      "n":len(df),
      "accuracy": accuracy_score(y_true, y_pred),
      "macro_f1": f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)
      }
  return summary

def summarize_per_class_stats(df):
    y_true, y_pred = predict_output_df(df)
    precision, recall, per_class_f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(LABELS))), average=None, zero_division=0)
    per_class_df = pd.DataFrame({
        "label": LABELS,
        "precision": precision,
        "recall": recall,
        "f1": per_class_f1,
        "support": support,
    })

    report = classification_report(y_true, y_pred, labels=list(range(len(LABELS))), target_names=LABELS, output_dict=True)

    print(report)

    return per_class_df


In [32]:
print("SMP Chinese Test Summaries")
cn_summary = summarize_test_set(cn_test_df, "SMP Chinese Test")
cn_per_class_summary = summarize_per_class_stats(cn_test_df)

print("\nGoEmotions English Test Summaries")
en_summary = summarize_test_set(en_test_df, "GoEmotions English Test")
en_per_class_summary = summarize_per_class_stats(en_test_df)

print("\nCombined Multilingual Test Summaries")
combined_summary = summarize_test_set(test_df, "Combined Multilingual Test")
combined_per_class_summary = summarize_per_class_stats(test_df)


SMP Chinese Test Summaries


Map:   0%|          | 0/3360 [00:00<?, ? examples/s]

Map:   0%|          | 0/3360 [00:00<?, ? examples/s]

{'joy': {'precision': 0.7709677419354839, 'recall': 0.7091988130563798, 'f1-score': 0.7387944358578052, 'support': 674.0}, 'anger': {'precision': 0.8205405405405405, 'recall': 0.7939330543933054, 'f1-score': 0.8070175438596491, 'support': 956.0}, 'sadness': {'precision': 0.6014705882352941, 'recall': 0.6607431340872375, 'f1-score': 0.6297151655119323, 'support': 619.0}, 'fear': {'precision': 0.668918918918919, 'recall': 0.6827586206896552, 'f1-score': 0.6757679180887372, 'support': 145.0}, 'surprise': {'precision': 0.583011583011583, 'recall': 0.594488188976378, 'f1-score': 0.5886939571150097, 'support': 254.0}, 'neutral': {'precision': 0.8379120879120879, 'recall': 0.8567415730337079, 'f1-score': 0.8472222222222222, 'support': 712.0}, 'accuracy': 0.7458333333333333, 'macro avg': {'precision': 0.7138035767589846, 'recall': 0.7163105640394439, 'f1-score': 0.7145352071092259, 'support': 3360.0}, 'weighted avg': {'precision': 0.7494199109703905, 'recall': 0.7458333333333333, 'f1-score': 0

Map:   0%|          | 0/2084 [00:00<?, ? examples/s]

Map:   0%|          | 0/2084 [00:00<?, ? examples/s]

{'joy': {'precision': 0.7708333333333334, 'recall': 0.7956989247311828, 'f1-score': 0.783068783068783, 'support': 93.0}, 'anger': {'precision': 0.5492957746478874, 'recall': 0.5954198473282443, 'f1-score': 0.5714285714285714, 'support': 131.0}, 'sadness': {'precision': 0.7386363636363636, 'recall': 0.6372549019607843, 'f1-score': 0.6842105263157895, 'support': 102.0}, 'fear': {'precision': 0.8059701492537313, 'recall': 0.8307692307692308, 'f1-score': 0.8181818181818182, 'support': 65.0}, 'surprise': {'precision': 0.6986301369863014, 'recall': 0.5862068965517241, 'f1-score': 0.6375, 'support': 87.0}, 'neutral': {'precision': 0.9165636588380717, 'recall': 0.9234122042341221, 'f1-score': 0.9199751861042184, 'support': 1606.0}, 'accuracy': 0.8661228406909789, 'macro avg': {'precision': 0.7466549027826148, 'recall': 0.7281270009292148, 'f1-score': 0.7357274808498634, 'support': 2084.0}, 'weighted avg': {'precision': 0.8657179814217976, 'recall': 0.8661228406909789, 'f1-score': 0.86544936680

Map:   0%|          | 0/5444 [00:00<?, ? examples/s]

Map:   0%|          | 0/5444 [00:00<?, ? examples/s]

{'joy': {'precision': 0.770949720670391, 'recall': 0.7196870925684485, 'f1-score': 0.7444369521240728, 'support': 767.0}, 'anger': {'precision': 0.7844423617619494, 'recall': 0.7700091996320148, 'f1-score': 0.7771587743732591, 'support': 1087.0}, 'sadness': {'precision': 0.6171875, 'recall': 0.6574202496532594, 'f1-score': 0.6366689053055742, 'support': 721.0}, 'fear': {'precision': 0.7116279069767442, 'recall': 0.7285714285714285, 'f1-score': 0.72, 'support': 210.0}, 'surprise': {'precision': 0.608433734939759, 'recall': 0.592375366568915, 'f1-score': 0.600297176820208, 'support': 341.0}, 'neutral': {'precision': 0.8921568627450981, 'recall': 0.9029335634167386, 'f1-score': 0.8975128644939966, 'support': 2318.0}, 'accuracy': 0.7918809698750918, 'macro avg': {'precision': 0.7307996811823236, 'recall': 0.7284994834018009, 'f1-score': 0.7293457788528518, 'support': 5444.0}, 'weighted avg': {'precision': 0.7924204339478582, 'recall': 0.7918809698750918, 'f1-score': 0.7919046965358176, 'su

In [33]:
test_metrics = pd.DataFrame([
    cn_summary,
    en_summary,
    combined_summary,
])

display(test_metrics)

,model,dataset,n,accuracy,macro_f1
0,Finetuned-mBERT,SMP Chinese Test,3360,0.745833,0.714535
1,Finetuned-mBERT,GoEmotions English Test,2084,0.866123,0.735727
2,Finetuned-mBERT,Combined Multilingual Test,5444,0.791881,0.729346


In [35]:
per_class_metrics = pd.concat(
    [cn_per_class_summary, en_per_class_summary, combined_per_class_summary],
    ignore_index=True,
)

display(per_class_metrics)

,label,precision,recall,f1,support
0,joy,0.770968,0.709199,0.738794,674
1,anger,0.820541,0.793933,0.807018,956
2,sadness,0.601471,0.660743,0.629715,619
3,fear,0.668919,0.682759,0.675768,145
4,surprise,0.583012,0.594488,0.588694,254
5,neutral,0.837912,0.856742,0.847222,712
6,joy,0.770833,0.795699,0.783069,93
7,anger,0.549296,0.595420,0.571429,131
8,sadness,0.738636,0.637255,0.684211,102
9,fear,0.805970,0.830769,0.818182,65


In [39]:
# Save to files just in case
test_metrics_path = results_folder / "mbert_test_metrics.csv"
per_class_path = results_folder / "mbert_per_class_metrics.csv"

test_metrics.to_csv(test_metrics_path, index=False)
per_class_metrics.to_csv(per_class_path, index=False)

print("Saved:", test_metrics_path)
print("Saved:", per_class_path)

Saved: /content/data/results/mbert_test_metrics.csv
Saved: /content/data/results/mbert_per_class_metrics.csv


## 8. Compare fine-tuned mBERT against the baseline

In [37]:
# Table we can report on
comparison = pd.concat(
    [baseline_metrics[["model", "dataset", "accuracy", "macro_f1"]],
     test_metrics[["model", "dataset", "accuracy", "macro_f1"]]],
    ignore_index=True,
).sort_values(
    ["dataset", "model"]
).reset_index(drop=True)

display(comparison)

,model,dataset,accuracy,macro_f1
0,Finetuned-mBERT,Combined Multilingual Test,0.791881,0.729346
1,majority_class,Combined Multilingual Test,0.425790,0.099545
2,Finetuned-mBERT,GoEmotions English Test,0.866123,0.735727
3,majority_class,GoEmotions English Test,0.770633,0.145077
4,Finetuned-mBERT,SMP Chinese Test,0.745833,0.714535
5,majority_class,SMP Chinese Test,0.211905,0.058284


## End of notebook notes:

- Best model as of right now is `checkpoint-7134`
In the next step we can simply use the best_checkpoint without having to re-tune mBERT
